In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from utils import csv_downloader

In [13]:
df_order = csv_downloader(URL = 'https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/rfm/orders.csv',
                    name = 'orders',
                    path = '../data/rfm')

orders saved in ../data/rfm | shape: (4906, 3)


In [14]:
df_order = pd.read_csv('../data/rfm/orders.csv', parse_dates = ['order_date'])

In [15]:
# RFM
df_order.info()

<class 'pandas.DataFrame'>
RangeIndex: 4906 entries, 0 to 4905
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  4906 non-null   str           
 1   order_date   4906 non-null   datetime64[us]
 2   revenue      4906 non-null   int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 115.1 KB


In [16]:
fig = px.histogram(data_frame=df_order,
             x = 'revenue',
             title = 'Revenue Distribution',
             nbins = 40
             )
fig.update_layout(
    xaxis_title = 'Revenue',
    yaxis_title = 'Count'
)
fig.show()

### RFM

In [17]:
#Loading Packages and Data
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)
px.defaults.template = "plotly_white"

In [23]:
orders = pd.read_csv('../data/rfm/orders.csv', parse_dates=['order_date'])
orders.head()

,customer_id,order_date,revenue
0,Mr. Brion Stark Sr.,2004-12-20,32
1,Ethyl Botsford,2005-05-02,36
2,Hosteen Jacobi,2004-03-06,116
3,Mr. Edw Frami,2006-03-15,99
4,Josef Lemke,2006-08-14,76


In [26]:
fix = px.histogram(
    orders,
    x='revenue',
    nbins=30,
    title='Histogram of Revenue'
)

fig.update_layout(
    xaxis_title='Revenue',
    yaxis_title='Count'
)

fig.show()

In [31]:
#R, F, M Scores

max_date = orders['order_date'].max()

rfm = orders.groupby('customer_id').agg(
    {
        'order_date': lambda date: (max_date - date.max()).days,
        'customer_id': lambda num: len(num),
        'revenue': lambda price: price.sum()
    }
)

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.reset_index(inplace=True)
rfm.head()

,customer_id,Recency,Frequency,Monetary
0,Abbey O'Reilly DVM,204,6,472
1,Add Senger,139,3,340
2,Aden Lesch Sr.,193,4,405
3,Admiral Senger,131,5,448
4,Agness O'Keefe,89,9,843


In [32]:
rfm['R'] = pd.qcut(rfm['Recency'], 4, ['4', '3', '2', '1'])
rfm['F'] = pd.qcut(rfm['Frequency'], 4, ['1', '2', '3', '4'])
rfm['M'] = pd.qcut(rfm['Monetary'], 4, ['1', '2', '3', '4'])

In [33]:
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M
0,Abbey O'Reilly DVM,204,6,472,3,3,3
1,Add Senger,139,3,340,3,1,2
2,Aden Lesch Sr.,193,4,405,3,2,2
3,Admiral Senger,131,5,448,4,2,3
4,Agness O'Keefe,89,9,843,4,4,4


In [34]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("Recency", "Frequency", "Monetary")
)

fig.add_trace(
    go.Histogram(x=rfm["Recency"], nbinsx=30, name="Recency"),
    row=1,
    col=1
)

fig.add_trace(
    go.Histogram(x=rfm["Frequency"], nbinsx=30, name="Frequency"),
    row=1,
    col=2
)

fig.add_trace(
    go.Histogram(x=rfm["Monetary"], nbinsx=30, name="Monetary"),
    row=1,
    col=3
)

fig.update_layout(
    title="",
    showlegend=False
)

fig.update_xaxes(title_text="Recency", row=1, col=1)
fig.update_xaxes(title_text="Frequency", row=1, col=2)
fig.update_xaxes(title_text="Monetary", row=1, col=3)

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=3)

fig.show()

In [37]:
rfm['RFM_Score'] = rfm.R.astype(int) + rfm.F.astype(int) + rfm.M.astype(int)
rfm['RFM_Segment'] = rfm.R.astype(str) + rfm.F.astype(str) + rfm.M.astype(str)

rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment
0,Abbey O'Reilly DVM,204,6,472,3,3,3,9,333
1,Add Senger,139,3,340,3,1,2,6,312
2,Aden Lesch Sr.,193,4,405,3,2,2,7,322
3,Admiral Senger,131,5,448,4,2,3,9,423
4,Agness O'Keefe,89,9,843,4,4,4,12,444


In [38]:
#Finding top customers
rfm = rfm.sort_values('RFM_Segment', ascending=False)
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment
345,Exie Gutmann,102,7,651,4,4,4,12,444
106,Cameron Abbott DDS,107,8,723,4,4,4,12,444
693,Mr. Semaj Sauer,30,8,806,4,4,4,12,444
442,Jeanette Lind,40,10,1062,4,4,4,12,444
141,Cleveland Fay,87,8,790,4,4,4,12,444


In [40]:
#Number of RFM Segments
rfm_segments = rfm.groupby('RFM_Segment').count().reset_index(level=0)
rfm_segments.head()

,RFM_Segment,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score
0,111,90,90,90,90,90,90,90,90
1,112,31,31,31,31,31,31,31,31
2,113,6,6,6,6,6,6,6,6
3,121,11,11,11,11,11,11,11,11
4,122,29,29,29,29,29,29,29,29


In [44]:
fig = px.bar(
    rfm_segments.sort_values('customer_id'),
    x='customer_id',
    y='RFM_Segment',
    orientation='h',
    title='Number of Customers by RFM Segment'
)

fig.update_layout(
    xaxis_title='Number of Customers',
    yaxis_title='RFM Segment'
)

fig.show()

In [46]:
def naming(df):
    if df['RFM_Score'] >= 9:
        return 'Can\'t Loose Them'
    elif ((df['RFM_Score'] >= 8) and (df['RFM_Score'] < 9)):
        return 'Champions'
    elif ((df['RFM_Score'] >= 7) and (df['RFM_Score'] < 8)):
        return 'Loyal/Commited'
    elif ((df['RFM_Score'] >= 6) and (df['RFM_Score'] < 7)):
        return 'Potential'
    elif ((df['RFM_Score'] >= 5) and (df['RFM_Score'] < 6)):
        return 'Promising'
    elif ((df['RFM_Score'] >= 4) and (df['RFM_Score'] < 5)):
        return 'Requires Attention'
    else:
        return 'Demands Activation'

In [47]:
#Create Segment_Name coulmn
rfm['Segment_Name'] = rfm.apply(naming, axis=1)
rfm.head(5)

,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment,Segment_Name
345,Exie Gutmann,102,7,651,4,4,4,12,444,Can't Loose Them
106,Cameron Abbott DDS,107,8,723,4,4,4,12,444,Can't Loose Them
693,Mr. Semaj Sauer,30,8,806,4,4,4,12,444,Can't Loose Them
442,Jeanette Lind,40,10,1062,4,4,4,12,444,Can't Loose Them
141,Cleveland Fay,87,8,790,4,4,4,12,444,Can't Loose Them


In [60]:
#Segment Distribution
grouped_by = rfm.groupby('Segment_Name').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(1)

In [61]:
#Segments’ Distribution
grouped_by

Recency Frequency Monetary      
                      mean      mean     mean count
Segment_Name                                       
Can't Loose Them     166.6       7.2    707.2   335
Champions            212.0       5.0    489.6   114
Demands Activation   600.5       1.9    151.0    90
Loyal/Commited       275.5       4.7    429.3   147
Potential            286.8       3.9    352.1   132
Promising            361.4       3.4    294.8    97
Requires Attention   460.7       2.8    246.0    80

In [62]:
grouped_by.columns = ['RecencyMean', 'FrequencyMean', 'MonetaryMean', 'Count']
grouped_by = grouped_by.reset_index()

grouped_by

,Segment_Name,RecencyMean,FrequencyMean,MonetaryMean,Count
0,Can't Loose Them,166.6,7.2,707.2,335
1,Champions,212.0,5.0,489.6,114
2,Demands Activation,600.5,1.9,151.0,90
3,Loyal/Commited,275.5,4.7,429.3,147
4,Potential,286.8,3.9,352.1,132
5,Promising,361.4,3.4,294.8,97
6,Requires Attention,460.7,2.8,246.0,80


In [63]:
fig = px.treemap(
    grouped_by,
    path=['Segment_Name'],
    values='Count',
    color='MonetaryMean',
    hover_data=['RecencyMean', 'FrequencyMean', 'MonetaryMean'],
    title='RFM Segments'
)

fig.show()